

Experiment1 execution guide · MD
# Track A — Experiment 1 (T1-R) Execution Guide
**Team of 2, Windows + VS Code, shared `regenesis` GitHub repo, one branch per track**
 
This guide covers three things: how to set up your local workspace against the shared repo, how you and your teammate split the work, and the exact phase-by-phase plan for T1-R (the motor-imagery reproduction experiment).
 
**Repo model, stated up front:** `regenesis` is one shared repository. Each track works on its own long-lived branch — `track-a`, `track-b`, `track-c`, etc. — which someone (a lead, or whoever owns the repo) will add you and your teammate to as collaborators. Nobody on Track A commits directly to `main`; `main` is the integration point across tracks, not your working branch. `eval/` (splitters, surrogates, statistical utilities) is owned by Track E — Track A consumes it, never writes to it.
 
---
 
## Part 1 — Git & Workspace Setup
 
### 1.1 Getting onto `track-a`
 
Once you've been added as a collaborator on `regenesis`, clone it and check out `track-a`:
 
```bash
git clone https://github.com/<org>/regenesis.git
cd regenesis
git fetch origin
git checkout -b track-a origin/track-a
```
 
If `track-a` doesn't exist on the remote yet (i.e., you're the first person from the track to touch the repo), create it off `main` and push it:
 
```bash
git checkout main
git pull origin main
git checkout -b track-a
git push -u origin track-a
```
 
Both you and your teammate should end up with a local `track-a` branch tracking `origin/track-a`. This is your team's integration branch for all of Track A's work.
 
### 1.2 `.gitignore`
 
Create this in the repo root if it isn't already there, and commit it directly to `track-a` (low-risk, needed immediately by both of you):
 
```gitignore
# Python
__pycache__/
*.pyc
.venv/
venv/
*.egg-info/
 
# Jupyter
.ipynb_checkpoints/
 
# Data & large artifacts — never commit raw data
data/raw/
data/*.mat
data/*.fif
data/*.edf
*.npz
*.npy
 
# Results caches (keep the small deliverables, not intermediate cache)
results/*/cache/
 
# Environment / secrets
.env
*.lock.local
 
# OS / editor
.vscode/*
!.vscode/settings.json
.DS_Store
Thumbs.db
```
 
```bash
git add .gitignore
git commit -m "chore: add .gitignore for track-a"
git push origin track-a
```
 
`env.lock` (the pinned dependency file) is **not** ignored — it's part of the reproducibility floor (§8.8 of the manual) and gets committed as a real deliverable inside each result directory.
 
### 1.3 Directory structure for T1-R
 
The manual specifies the full Track A structure. For Experiment 1 specifically, here's what to build now versus what stays an empty placeholder for later experiments:
 
```
regenesis/
├── data/
│   ├── bci_iv_2a.py          # BUILD NOW — loader for BCI-IV-2a
│   ├── way_eeg_gal.py        # placeholder — needed for E2/E3, not T1-R
│   └── cards/
│       └── bci_iv_2a.md      # BUILD NOW — dataset card
├── preprocessing/
│   ├── eeg_filters.py        # BUILD NOW — band-pass 8–30 Hz
│   ├── epoching.py           # BUILD NOW — epoch per source paper's timing
│   ├── eeg_artifact.py       # placeholder — not needed for T1-R
│   └── features_eeg.py       # placeholder — not needed for T1-R
├── eeg/
│   ├── decoders_linear.py    # BUILD NOW — CSP + sLDA
│   ├── decoders_riemann.py   # optional alt. — use whichever family matches the source paper
│   ├── decoders_eegnet.py    # placeholder — not needed for T1-R
│   ├── montage.py            # placeholder — E2 only
│   ├── myogenic.py           # placeholder — E2 only
│   └── anticipation.py       # placeholder — E3 only
├── eval/                     # Track E owns this — Track A consumes, never writes
│   ├── splits.py
│   ├── surrogates.py
│   └── stats.py
├── experiments/
│   └── t1r_reproduction.yaml # BUILD NOW — orchestrates the full pipeline
├── configs/
│   └── t1r_base.yaml         # seeds, band edges, epoch window — pinned
├── prereg/
│   └── PR-2026-01.md         # WRITE FIRST, before touching data
├── results/
│   └── PR-2026-01/           # populated by the pipeline run, not by hand
├── ledger/                   # not required to close out T1-R (§5.1 has no decision gate)
└── tests/
    └── test_pipeline_smoke.py
```
 
Scaffold what you need now:
 
```bash
mkdir data\cards, preprocessing, eeg, eval, experiments, configs, prereg, results, ledger, tests
type nul > data\bci_iv_2a.py
type nul > data\cards\bci_iv_2a.md
type nul > preprocessing\eeg_filters.py
type nul > preprocessing\epoching.py
type nul > eeg\decoders_linear.py
type nul > experiments\t1r_reproduction.yaml
type nul > configs\t1r_base.yaml
type nul > prereg\PR-2026-01.md
type nul > tests\test_pipeline_smoke.py
```
*(`type nul >` is CMD's equivalent of `touch`; if you're using Git Bash, use `touch` directly. Don't create the `eval/` files — that folder is Track E's.)*
 
```bash
git add .
git commit -m "chore: scaffold T1-R directory structure per Track A manual §8.1"
git push origin track-a
```
 
### 1.4 Workflow rules
 
| Action | Rule |
|---|---|
| **Pull** | Every time you sit down to work, before writing anything: `git pull origin track-a`. |
| **Commit** | Per logical unit of work — one working function, one passing test, one fixed bug. Reference the pre-registration ID once it exists: `git commit -m "feat(PR-2026-01): add BCI-IV-2a loader"`. Don't commit broken code to `track-a` directly — do it on a feature branch. |
| **Push** | At least once per session, always before ending the day. An unpushed commit is invisible to your teammate and unrecoverable if your machine dies. |
| **Open a PR** | As soon as a feature branch has something reviewable, open it as a **draft PR** so your teammate can see direction early, then mark it ready when done. Base branch is `track-a`, not `main` — GitHub defaults to `main`, so check that dropdown every single time. |
| **Merge** | Squash-merge into `track-a` once your teammate has actually run the code, not just read the diff. Delete the branch after merge. |
| **Resolve conflicts** | Prefer `git pull --rebase origin track-a` over merge commits to keep history linear. On conflict, resolve in VS Code's inline editor (`<<<<<<<` / `=======` / `>>>>>>>`), then `git add <file>` and `git rebase --continue`. Never blindly accept "ours" or "theirs" without reading both sides. |
| **Sync `track-a` with `main`** | This is how Track E's `eval/` (and anything else from other tracks) reaches you. Confirm with your lead whether this integration step is yours to do or theirs — merging `main` into `track-a` can pull in other tracks' in-progress work, so it shouldn't be done casually by either of you without knowing who's responsible for it. |
 
---
 
## Part 2 — Team Work Division
 
### 2.1 Branching strategy: feature branches off `track-a`
 
Within Track A, branch off `track-a` for each logical piece of work, using the manual's naming convention (§8.3):
 
```
track-a/t1r-data-loader
track-a/t1r-preprocessing
track-a/t1r-decoder-csp-slda
track-a/t1r-experiment-orchestration
```
 
With only two people, pure file separation (you never touch my files, I never touch yours) feels safer than it is — it breaks the moment you need to change something shared, like the config schema both your scripts read from. Feature branches plus PRs into `track-a` give you parallel work without breaking the team's integration branch, and a forced second reviewer on every merge. That review step matters concretely here: a leaky split or a window-level split is exactly the class of bug the manual expects to be caught at review, not discovered later.
 
```bash
git checkout track-a
git pull origin track-a
git checkout -b track-a/t1r-data-loader
# ... work, commit ...
git push -u origin track-a/t1r-data-loader
# Open PR on GitHub — base: track-a, compare: track-a/t1r-data-loader
# Request review from your teammate, merge once approved
```
 
### 2.2 Task split for T1-R
 
**Developer A — Data & Infrastructure**
- `data/bci_iv_2a.py`: loader, version hash recording
- `data/cards/bci_iv_2a.md`: dataset card, including the resolved download URL and access date
- Locates and documents the source paper's published number and its exact conditions (n, subjects, class definitions, chance level, split policy, feature family, classifier) — required before anyone writes modeling code
- `preprocessing/eeg_filters.py` (band-pass 8–30 Hz) and `preprocessing/epoching.py` (epoch timing matching the source paper, recorded explicitly)
- Drafts `prereg/PR-2026-01.md` (both of you sign off before data is touched)
- Repo scaffolding, `.gitignore`, `configs/t1r_base.yaml`

**Developer B — Modeling & Evaluation**
- `eeg/decoders_linear.py`: CSP feature extraction + sLDA classifier, matching the source paper's feature family
- Integrates `eval/splits.py`, `eval/surrogates.py`, `eval/stats.py` from Track E once available on `track-a` — if any are missing when needed, that's a cross-track blocker to raise, not something to build locally. A local copy of an `eval/` utility is explicitly called out in the manual as a review-blocking defect.
- `experiments/t1r_reproduction.yaml`: orchestration config wiring the pipeline end to end
- Structured logging per §8.4: band edges, filter order, epoch window, split policy, fold assignment, seed, runtime per stage
- Figure T1-R.1: reproduced-vs-published scatter with identity line and CIs
Both of you: write `methods.md`, `surrogates.md`, `claim.md`, `deviations.md` together. Don't let one person own the entire result directory — §0.3 of the manual requires someone *outside* the person who ran it to be able to reproduce it, and that starts with your own team.
 
### 2.3 Daily sync routine
 
- **Before starting:** `git pull origin track-a`, skim your teammate's recent commits/PRs.
- **End of day:** push everything, even if incomplete (on your own feature branch — never broken code on `track-a`). Leave a one-line note: what you did, what you saw, what's blocking you. This doubles as the format the manual already wants for the lab notebook (§9.2), so reuse it.
- **Before merging into `track-a`:** the other person actually runs the code, not just reads the diff.
- **Never work on the same file on the same branch simultaneously** without coordinating — that's the single biggest source of avoidable merge conflicts.
---
 
## Part 3 — Experiment 1 (T1-R) Step-by-Step Execution Plan
 
**Goal:** reproduce a published 2-class motor-imagery decoding accuracy on BCI-IV-2a under your own honest, block-wise-split evaluation, and report the delta from the published value.
 
### Phase 0 — Before touching any data
1. Locate the source paper you're reproducing. Extract and record: n, subjects, class definitions, chance level, split policy, feature family (CSP or Riemannian tangent-space), classifier (sLDA), and exact epoch timing relative to the cue.
2. Write `prereg/PR-2026-01.md`: the hypothesis (reproduce accuracy X within some tolerance), the mandatory controls (block-wise/session-wise split, label-shuffle surrogate), and what a wrong result looks like (reproducing far above published → suspect leakage; far below → check epoch timing/band definition first). Both teammates sign off. Pre-registrations are immutable after sign-off — deviations go in `deviations.md`, never edited back into this file.
3. Set up the environment and commit `env.lock` (e.g. `pip freeze > env.lock`, or a conda/poetry lockfile).
4. Download BCI-IV-2a from the BNCI Horizon 2020 repository / BCI Competition IV archive. Record the resolved URL and access date in `data/cards/bci_iv_2a.md`.
### Phase 1 — Data loading
5. `data/bci_iv_2a.py`: load the raw corpus into a canonical schema, compute and record a version hash (`data.sha256`).
6. Verify subject count, session structure, class labels, and trial counts against the competition documentation.
7. Required visualizations before modeling (§6.2): per-class ERD time-frequency maps over C3/C4, per-subject class balance.
### Phase 2 — Preprocessing
8. Band-pass filter 8–30 Hz (`preprocessing/eeg_filters.py`).
9. Epoch relative to the cue, using the source paper's exact timing — recorded explicitly in the config, not just in your head.
### Phase 3 — Feature extraction
10. CSP or Riemannian tangent-space features (`eeg/decoders_linear.py` or `decoders_riemann.py`), matching whichever family the source paper used.
### Checkpoint — stop-and-debug gate
11. Plot the first two CSP/tangent-space components per class. **If classes aren't visibly separated for your best subjects, stop and debug before proceeding to classification.** Don't tune a classifier on top of broken features.
### Phase 4 — Classification & evaluation
12. sLDA classifier on top of the extracted features.
13. Block-wise split from `eval/splits.py` — session-wise, since BCI-IV-2a provides distinct sessions. Never window-level.
14. Label-shuffle surrogate from `eval/surrogates.py`, run alongside the real result, not as an afterthought.
15. Per-subject accuracy with bootstrap percentile CIs (≥1000 resamples) via `eval/stats.py`. Permutation test against the label-shuffle null.
### Phase 5 — Results & figure
16. Produce Figure T1-R.1: per-subject reproduced-vs-published accuracy scatter, with the identity line and CIs.
17. Interpret: points near the identity line → pipeline validated. Systematically below → your evaluation is stricter (report the gap). Above → investigate leakage before celebrating, check the split first.
### Phase 6 — Close out the result directory
18. Assemble `results/PR-2026-01/`:
    - `figure.png` (Figure T1-R.1)
    - `claim.md` — one sentence, with scope, nothing else (e.g. *"We reproduce 2-class motor-imagery decoding at X% (median, 95% CI [..]) across n subjects of BCI-IV-2a, session-wise splits, CSP+sLDA, a delta of Y points from the published Z%."*)
    - `methods.md` — full pipeline description
    - `surrogates.md` — the label-shuffle result
    - `env.lock`, `data.sha256`, `seed.txt`
    - `deviations.md` — anything that differed from the pre-registration, dated
19. **Reproducibility test:** have the teammate who didn't run the pipeline clone fresh and regenerate the figure in under 30 minutes. This — not "it ran on my machine" — is the actual bar for done.
### Milestones & deliverables checklist
 
- [ ] `prereg/PR-2026-01.md` signed by both teammates, before data was touched
- [ ] `results/PR-2026-01/` complete: `figure.png`, `claim.md`, `methods.md`, `surrogates.md`, `env.lock`, `data.sha256`, `seed.txt`, `deviations.md`
- [ ] `data/cards/bci_iv_2a.md` with resolved URL and access date
- [ ] Per-subject accuracy reported with bootstrap CIs, delta from published value stated
- [ ] Label-shuffle surrogate run and reported alongside the real result
- [ ] Block-wise (session-wise) split used — verified, not assumed
- [ ] Teammate who didn't run it regenerated the figure from the repo in under 30 minutes
- [ ] All code merged into `track-a` via reviewed PRs, no broken commits on `track-a`
- [ ] A written hypothesis for whatever discrepancy exists between your number and the published one
That last item is the actual scientific output of T1-R: not just a number, but a number plus a stated reason for why it differs from the published one, if it does.
 

